# 1. Header, Citations, Purpose & Topic Bridges

**Topic:** Differentiable Optimization: OptNet, KKT Conditions & Implicit Layers
**Category:** 06_smooth_and_physics_informed_ai

### Citations & Core References
- Amos, B., & Kolter, J. Z. (2017). *OptNet: Differentiable Optimization as a Layer in Neural Networks*. ICML.
- Agrawal, A., Amos, B., Barratt, S., Boyd, S., Diamond, S., & Kolter, J. Z. (2019). *Differentiable Convex Optimization Layers*. NeurIPS.
- Gould, S., Hartley, R., & Campbell, D. (2016). *On Differentiating Parameterized Argmin and Argmax Problems with Application to Bi-level Optimization*. arXiv:1607.05447.
- Boyd, S., & Vandenberghe, L. (2004). *Convex Optimization*. Cambridge University Press.

## 2. Mathematical Foundations & Governing Equations

### Quadratic Program (QP) Formulation
$$\min_{z} \frac{1}{2} z^T Q z + p^T z \quad \text{s.t.} \quad A z = b, \quad G z \le h$$
where $Q \succ 0$ is strictly positive definite.

### KKT System & Stationarity
For equality-constrained QPs, the optimality conditions form a linear system:
$$\begin{bmatrix} Q & A^T \\ A & 0 \end{bmatrix} \begin{bmatrix} z^\star \\ \nu^\star \end{bmatrix} = \begin{bmatrix} -p \\ b \end{bmatrix}.$$

### Implicit Function Theorem Backpropagation (OptNet)
Differentiating the KKT conditions with respect to parameters yields the adjoint system:
$$\begin{bmatrix} Q & A^T \\ A & 0 \end{bmatrix} \begin{bmatrix} d_z \\ d_\nu \end{bmatrix} = \begin{bmatrix} -\nabla_{z^\star}\mathcal{L} \\ 0 \end{bmatrix} \implies \nabla_p \mathcal{L} = -d_z, \quad \nabla_b \mathcal{L} = -d_\nu.$$

### Log-Barrier Interior Point Approximation
Smooth inequality constraints via logarithmic penalty:
$$\phi_t(z) = \frac{1}{2} z^T Q z + p^T z - \frac{1}{t} \sum_{i=1}^K \log(h_i - g_i^T z).$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import torch
import numpy as np

torch.manual_seed(42)
B, D, M, K = 4, 3, 1, 2

# Positive definite Q: M^T M + 0.2 I
m_q = torch.randn(D, D) * 0.4
Q = m_q.t() @ m_q + 0.2 * torch.eye(D)
p = torch.randn(B, D) * 0.5
A = torch.randn(M, D) * 0.5
b = torch.randn(B, M) * 0.2

print(f"Batch B: {B}, Dim D: {D}, Eq constraints M: {M}")
print(f"Condition number of Q: {torch.linalg.cond(Q):.4f}")

In [ ]:
# 4. First-Principles Implementation & Native Parity
# 1. Equality-Constrained QP Solve via Block KKT Inversion
KKT = torch.zeros(D + M, D + M)
KKT[:D, :D] = Q
KKT[:D, D:] = A.t()
KKT[D:, :D] = A

rhs = torch.cat([-p, b], dim=-1)
sol = torch.linalg.solve(KKT, rhs.t()).t()
z_eq = sol[:, :D]
nu_eq = sol[:, D:]

# 2. OptNet Implicit Backward Step
grad_loss_z = torch.randn(B, D) * 0.3
rhs_adj = torch.cat([-grad_loss_z, torch.zeros(B, M)], dim=-1)
sol_adj = torch.linalg.solve(KKT, rhs_adj.t()).t()
d_z = sol_adj[:, :D]
d_nu = sol_adj[:, D:]
grad_p = -d_z
grad_b = -d_nu

# 3. Log-Barrier Inequality QP Solve
G = torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
h_bound = torch.tensor([[1.5, 1.5]]).expand(B, K)
z_barrier = torch.zeros(B, D)
t_barrier = 5.0
eta = 0.08
for _ in range(20):
    slacks = torch.clamp(h_bound - z_barrier @ G.t(), min=1e-3)
    lambda_dual = 1.0 / (t_barrier * slacks)
    grad_barrier = lambda_dual @ G
    grad_total = z_barrier @ Q + p + grad_barrier
    z_barrier = z_barrier - eta * grad_total

final_slacks = torch.clamp(h_bound - z_barrier @ G.t(), min=1e-4)
lambda_barrier = 1.0 / (t_barrier * final_slacks)

print(f"Equality QP solution norm: {torch.norm(z_eq):.4f}, Dual multiplier norm: {torch.norm(nu_eq):.4f}")
print(f"Adjoint grad_p norm: {torch.norm(grad_p):.4f}, Barrier solution norm: {torch.norm(z_barrier):.4f}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification
# Invariant 1: Primal feasibility ||A z - b|| < 1e-5
primal_res = torch.max(torch.abs(z_eq @ A.t() - b)).item()
assert primal_res < 1e-5, f"Primal feasibility violated: {primal_res:.2e}"

# Invariant 2: Stationarity residual ||Q z + p + A^T nu|| < 1e-5
stat_res = torch.max(torch.abs(z_eq @ Q + p + nu_eq @ A)).item()
assert stat_res < 1e-5, f"Stationarity violated: {stat_res:.2e}"

# Invariant 3: Adjoint KKT identity
adj_stat_res = torch.max(torch.abs(d_z @ Q + d_nu @ A - (-grad_loss_z))).item()
assert adj_stat_res < 1e-5, f"Adjoint stationarity violated: {adj_stat_res:.2e}"

print(f"All invariants verified! Primal res: {primal_res:.2e}, Stationarity: {stat_res:.2e}")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

iters = 100
t0 = time.perf_counter()
for _ in range(iters):
    _ = torch.linalg.solve(KKT, rhs.t()).t()
dt = (time.perf_counter() - t0) / iters * 1000.0
print(f"KKT solve latency: {dt:.3f} ms / step")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation
# Pathological: Rank-deficient constraint matrix A causes singular KKT matrix and numerical collapse
A_bad = torch.zeros(M, D) # Zero row rank
KKT_bad = torch.zeros(D + M, D + M)
KKT_bad[:D, :D] = Q
try:
    _ = torch.linalg.solve(KKT_bad, rhs.t())
    print("Unexpected solve on singular matrix")
except Exception as e:
    print(f"Pathological singular KKT properly caught: {type(e).__name__}")

# Corrected: Adding Tikhonov regularization epsilon * I to the dual block ensures bounded invertibility
epsilon = 1e-5
KKT_reg = KKT_bad.clone()
KKT_reg[D:, D:] = -epsilon * torch.eye(M)
sol_reg = torch.linalg.solve(KKT_reg, rhs.t()).t()
print(f"Corrected regularized solution norm: {torch.norm(sol_reg):.4f}")

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Declarative Architecture Layers**: Embedding convex optimization programs directly into networks enforces hard physical, economic, or safety constraints.
2. **Exact Implicit Gradients**: OptNet differentiates through KKT stationarity, bypassing unrolled iterations with zero activation memory overhead.
3. **Interior Point Duality**: Log-barrier smoothing bridges combinatorial inequality switching into smooth continuous end-to-end backpropagation.